# Novae Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 03.12.2024
- Date of Last Modification: 02.04.2026 (Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the novae uv environment, installable from ```../../envs/novae.sh```
- This notebook follows the tutorial from https://mics-lab.github.io/novae/tutorials/main_usage/.

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import math
import os
import warnings

import anndata as ad
import pandas as pd
import matplotlib as mpl
import novae
import numpy as np
import scanpy as sc
import scipy.sparse as sp
import sklearn.metrics as skm
import squidpy as sq
from novae._constants import Keys
from scib_metrics import ilisi_knn
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'

### 1.3 Define Config

In [ ]:
n_pcs = None
n_neighs = 10
spot_sizes = {'MERFISH': 20,
              'STARmap PLUS': 100}

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

pd.options.future.infer_string = False

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/Novae', exist_ok=True)

In [ ]:
LABEL_MAP = {
    "merfish":      "MERFISH",
    "starmap_plus": "STARmap PLUS",
    "STARmap_PLUS": "STARmap PLUS",
    "MERFISH":      "MERFISH",
}

### 1.5 Define Functions

In [ ]:
def _combine_neighbor_graphs(adata: ad.AnnData,
                             n_neighs: int | None,
                             radius: float | None,
                             delaunay: bool,
                             include_self_loop: bool,
                             ) -> ad.AnnData:
    """
    Helper function to combine neighbor graphs computed using different
    methods.
    """
    if n_neighs is not None:
        # Compute knn spatial neighborhood graph
        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                n_neighs=n_neighs,
                                set_diag=include_self_loop,
                                ) 
        knn_conn = adata.obsp['spatial_connectivities']  
    if radius is not None:
        # Compute spatial neighborhood graph with radius
        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                radius=radius,
                                set_diag=include_self_loop,
                                )
        radius_conn = adata.obsp['spatial_connectivities'] 
        if n_neighs is not None:
            adata.obsp[
                'spatial_connectivities'] = knn_conn.multiply(
                    adata.obsp['spatial_connectivities'])
    if delaunay:
        # Compute spatial neighborhood graph with delaunay triangulation
        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                delaunay=True,
                                set_diag=include_self_loop,
                                )
        if n_neighs is not None:
            adata.obsp[
                'spatial_connectivities'] = knn_conn.multiply(
                    adata.obsp['spatial_connectivities'])
        if radius is not None:
            adata.obsp[
                'spatial_connectivities'] = radius_conn.multiply(
                    adata.obsp['spatial_connectivities'])

    return adata


def construct_neighbor_graph(adata: ad.AnnData,
                             n_neighs: int | None = None,
                             radius: float | None = None,
                             delaunay: bool = True,
                             include_self_loop: bool = False,
                             batch_key: str | None = None,
                             compute_neighbor_counts: bool = False,
                             ) -> ad.AnnData:
    """
    Compute neighbor graph and optionally aggregate cell features across
    neighbors.

    Parameters
    ----------
    adata:
        AnnData object with spatial coordinates available in
        `adata.obsm['spatial']`.
    n_neighs:
        If specified, use `n_neighs` to compute the neighborhood graph.
        If 'radius' or 'delaunay' are also specified, an intersection
        neighborhood graph will be computed.
    radius:
        If specified, use `radius` to compute the neighborhood graph. If
        'n_neighs' or 'delaunay' are also specified, an intersection
        neighborhood graph will be computed.
    delaunay:
        If 'True', compute the neighborhood graph by delaunay
        triangulation. If 'n_neighs' or 'radius' are also specified, an
        intersection neighborhood graph will be computed.
    include_self_loop:
        If 'True', include cell itself in neighborhood graph.
    batch_key:
        Key in adata.obs where the batch is stored in case the AnnData
        contains multiple batches. This is important to specify if there
        are multiple batches in the AnnData to perform separate neighbor
        graph computation and then concatenate.
    compute_neighbor_counts:
        If 'True', aggregate counts across neighborhood and store in
        `adata.layers['X_neighborhood']`.

    Returns
    ----------
    adata:
        AnnData object with aggregated counts available in
        `adata.layers['X_neighborhood']`.
    """
    if batch_key is not None:
        # Get ordered batches
        first_idx = adata.obs.reset_index().groupby(batch_key).head(1).index
        batches = adata.obs.iloc[first_idx][batch_key].tolist()
        adata_batch_list = []

        for batch in batches:
            print(f"\nProcessing batch {batch}...")
            print("Loading data...")
            adata_batch = adata[adata.obs[batch_key] == batch]

            print("Computing spatial neighborhood graph...")
            adata_batch = _combine_neighbor_graphs(
                adata_batch,
                n_neighs,
                radius,
                delaunay,
                include_self_loop)
            adata_batch_list.append(adata_batch)

        # Block-diagonal concatenate graphs (disconnected components)
        adata.obsp['spatial_connectivities'] = sp.block_diag(
            [a.obsp['spatial_connectivities'] for a in adata_batch_list],
            format="csr",
        )    
    else:
        adata = _combine_neighbor_graphs(
            adata,
            n_neighs,
            radius,
            delaunay,
            include_self_loop)           
    
    if compute_neighbor_counts:
        adata.layers['X_neighborhood'] = (
            adata.obsp['spatial_connectivities'].T @ adata.X)

    return adata

## 2. Zero-shot Niche Identification

### 2.1 mmb0-1b_smb1-1b_1p (MERFISH + STARmap PLUS mouse brain shared panel)

#### 2.1.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'mmb0-1b_smb1-1b_1p'
batches = ['harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes',
           'harmonised_starmap_plus_mouse_cns_batch15_shared_genes']
niche_key = 'tissue_region'
batch_key = 'batch'
label_keys = [niche_key]
spot_sizes = [100, 20]

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            print(file)

            if file == 'harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['ccf_region_name']
                adata.obs[batch_key] = file.replace('h5ad', '')
            elif file == 'harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['Sub_molecular_tissue_region']
                adata.obs[batch_key] = file.replace('h5ad', '')                

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]
            print(adata.obs.columns)
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)

#### 2.1.2 Retrieve Embeddings

In [ ]:
novae.spatial_neighbors(adata, n_neighs=11, radius=None, delaunay=False)
#novae.spatial_neighbors(adata, radius=80)

In [ ]:
print('Computing spatial neighborhood...')
# Construct neighbor graph (overwrite novae graph which is not symmetric)
adata = construct_neighbor_graph(
    adata,
    n_neighs=n_neighs,
    radius=None,
    delaunay=None,
    include_self_loop=True,
    compute_neighbor_counts=False)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-mouse-0")

# Compute zero-shot embeddings
# This is not reproducible
model.compute_representations(adata, zero_shot=True)

In [ ]:
# Save updated AnnData
for col in adata.obs.columns:
    if isinstance(adata.obs[col].dtype, pd.CategoricalDtype):
        adata.obs[col] = adata.obs[col].astype(str).astype('category')

adata.obs['cell_id'] = adata.obs['cell_id'].astype(str)
adata.obs.index = adata.obs.index.astype(str)

def fix_arrow_strings(df):
    for col in df.columns:
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].astype(str).astype('category')
        elif isinstance(df[col].array, pd.arrays.ArrowExtensionArray):
            df[col] = df[col].astype(str)
    df.index = df.index.astype(str)
    return df

adata.obs = fix_arrow_strings(adata.obs)
adata.var = fix_arrow_strings(adata.var)

adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded.h5ad')

In [ ]:
for emb_key in ['novae_latent']:
    try:
        rsc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                         n_pcs=n_pcs, key_added=emb_key)
        rsc.tl.umap(adata, neighbors_key=emb_key)
    except NameError:
        sc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                        n_pcs=n_pcs, key_added=emb_key)
        sc.tl.umap(adata, neighbors_key=emb_key)

    n_pcs_string = f"_{n_pcs}_pcs" if n_pcs else ""

    categories = adata.obs[batch_key].cat.categories.tolist()
    assert len(categories) == 2, "Expected exactly 2 categories for batch_key"
    rename = dict(zip(sorted(categories), ["MERFISH", "STARmap PLUS"]))
    adata.obs[batch_key] = adata.obs[batch_key].map(rename).astype("category")

    sc.pl.umap(
        adata,
        color=batch_key,
        title="Technology",
        legend_loc="right margin",
        legend_fontsize=10,
        show=False,
    )



    plt.savefig(
        f"{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_{emb_key}_umap_by_{batch_key}{n_pcs_string}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()
    plt.close()

    for label_key in label_keys:
        sc.pl.umap(adata, color=label_key, show=False)
        plt.savefig(
            f"{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_{emb_key}_umap_by_{label_key}{n_pcs_string}.png",
            dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close()

In [ ]:
model.assign_domains(adata, level=15)

In [ ]:
batches = adata.obs[batch_key].unique()
adatas = [adata[adata.obs[batch_key] == b].copy() for b in batches]
adata = ad.concat(adatas)

In [ ]:
# Compute NMI with region annotations
mask = adata_concat.obs["Sub_molecular_tissue_region"].notna()
labels_true = adata_concat.obs.loc[mask, "Sub_molecular_tissue_region"]
labels_pred = adata_concat.obs.loc[mask, "novae_domains_15"]

nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
print(f"NMI: {nmi:.4f}")

mask = adata_concat.obs["ccf_region_name"].notna()
labels_true = adata_concat.obs.loc[mask, "ccf_region_name"]
labels_pred = adata_concat.obs.loc[mask, "novae_domains_15"]

nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
print(f"NMI (ccf_region_name): {nmi:.4f}")

In [ ]:
for sample in samples:
    adata_sample = adata[adata.obs[batch_key] == sample]
    sc.pl.spatial(
        adata_sample,
        color='novae_domains_15',
        palette=mapped_palette,
        spot_size=spot_sizes[sample],
        title=sample,
        legend_loc="right margin",
        legend_fontsize=10,
        show=False,
    )

    fig = plt.gcf()
    for ax in fig.axes:
        for child in ax.get_children():
            if isinstance(child, mpl.image.AxesImage):
                child.set_rasterized(True)
        for collection in ax.collections:
            collection.set_rasterized(True)

    plt.savefig(
        f"{benchmarking_artifacts_folder_path}/Novae/tissue_{sample}_by_novae_domains_15.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
novae.plot.domains(adatas)

In [ ]:
model.batch_effect_correction(adatas)
adata = ad.concat(adatas)

In [ ]:
for emb_key in ['novae_latent_corrected']:
    try:
        rsc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                         n_pcs=n_pcs, key_added=emb_key)
        rsc.tl.umap(adata, neighbors_key=emb_key)
    except NameError:
        sc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                        n_pcs=n_pcs, key_added=emb_key)
        sc.tl.umap(adata, neighbors_key=emb_key)

    n_pcs_string = f"_{n_pcs}_pcs" if n_pcs else ""
    categories = adata.obs[batch_key].cat.categories.tolist()
    assert len(categories) == 2, "Expected exactly 2 categories for batch_key"
    rename = dict(zip(sorted(categories), ["MERFISH", "STARmap PLUS"]))
    adata.obs[batch_key] = adata.obs[batch_key].map(rename).astype("category")

    sc.pl.umap(
        adata,
        color=batch_key,
        title="Technology",
        legend_loc="right margin",
        legend_fontsize=10,
        show=False,
    )
    for collection in plt.gca().collections:
        collection.set_rasterized(True)
    plt.savefig(
        f"{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_{emb_key}_umap_by_{batch_key}{n_pcs_string}.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

    for label_key in label_keys:
        sc.pl.umap(adata, color=label_key, show=False)
        for collection in plt.gca().collections:
            collection.set_rasterized(True)
        plt.savefig(
            f"{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_{emb_key}_umap_by_{label_key}{n_pcs_string}.svg",
            format="svg", bbox_inches="tight"
        )
        plt.show()
        plt.close()

In [ ]:
# Save updated AnnData
adata = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
# Compute iLISI
batch_key = "batch"
batch = adata.obs[batch_key].values

for embed_name, embed_key in [("uncorrected", Keys.REPR), ("corrected", Keys.REPR_CORRECTED)]:
    knn = pynndescent(adata.obsm[embed_key], n_neighbors=90)
    score = ilisi_knn(knn, batch)
    print(f"iLISI ({embed_name}): {score:.4f}")

In [ ]:
print("Number of unique batches:", adata.obs[batch_key].nunique())
print("Batch value counts:\n", adata.obs[batch_key].value_counts())
print()
print("Uncorrected embedding shape:", adata.obsm[Keys.REPR].shape)
print("Any NaNs (uncorrected):", np.isnan(adata.obsm[Keys.REPR]).any())
print("Any NaNs (corrected):", np.isnan(adata.obsm[Keys.REPR_CORRECTED]).any())
print()

# Also try unscaled iLISI to see the raw values
batch = adata.obs[batch_key].values
knn = pynndescent(adata.obsm[Keys.REPR], n_neighbors=90)
score_unscaled = ilisi_knn(knn, batch, scale=False)
print(f"Raw iLISI (uncorrected): {score_unscaled:.4f}")

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/Novae/{dataset_name}_embedded_clustered.h5ad')

## 3. Save Xenium Mouse Brain Datasets